In [1]:
# importando las librerias requeridas
import pandas as pd

En este caso empezaremos a trabajar con archivos en Excel, es necesario que instales el paquete *openpyxl* usando el comando `conda install openpyxl` en tu terminal de Anaconda. Asegúrate de realizar la instalación en el ambiente creado para el curso.

# ¿Qué estaciones de servicio ofrecen los mayores y menores precios?
A lo largo de este caso crearemos una función para determinar si el precio de la gasolina en una estación de servicio se desvía mucho del valor promedio en su región. También cargaremos nuestra primera base de datos para aplicar el análisis que hemos creado a más de 4493 estaciones de servicio a lo largo de todo el país.

## Resultado previsto de aprendizaje
Al terminar este caso, el estudiante debe estar en capacidad de utilizar ciclos como `if`, `elif`, `else` para hacer clasificaciones, y también ciclos como `for` o `while` para llevar a cabo tareas repetitivas.

---

## Contexto del Problema
A usted se le entregará una base de datos con los valores promedio de la gasolina y el diesel durante el mes de diciembre de 2020 para un total de 2811 estaciones de servicio en todo el país. Junto con los precios, cada estación de servicio está identificada por un nombre y también por un Departamento de ubicación en el país.

Adicionalmente, se le entregará información acerca del precio promedio, y también la desviación estándar, de la gasolina corriente y el diesel en los 10 Departamentos con el mayor número de estaciones de servicio en el país.

Su objetivo será clasificar como *Promedio (o)*, *Barata (-)* o *Cara (+)* cada una de las estaciones en los 10 Departamentos de los que tenemos información promedio, dependiendo de si el precio de la gasolina y el diesel es similar o significativamente menor o mayor al promedio del Departamento.

Los diccionarios siguientes contienen la información de los valores promedio, en pesos colombianos, del galón de gasolina corriente y el biodiesel para 10 departamentos del país:

In [4]:
gasolina_prom = {
    'ANTIOQUIA': 8246,
    'ATLANTICO': 8252,
    'BOGOTA D.C.': 8418,
    'CAQUETA': 7283,
    'CESAR': 8313,
    'CHOCO': 7196,
    'CUNDINAMARCA': 8543,
    'MAGDALENA': 7626,
    'QUINDIO': 8079,
    'SUCRE': 8258
    }

diesel_prom = {
    'ANTIOQUIA': 8288,
    'ATLANTICO': 8177,
    'BOGOTA D.C.': 8433,
    'CAQUETA': 7420,
    'CESAR': 8289,
    'CHOCO': 7386,
    'CUNDINAMARCA': 8584,
    'MAGDALENA': 7963,
    'QUINDIO': 8064,
    'SUCRE': 8312}

Adicional a los valores promedio, para nosotros va a ser importante el valor de la desviación estándar de los precios de la gasolina en estos departamentos:

In [5]:
gasolina_desv = {
    'ANTIOQUIA': 495,
    'ATLANTICO': 421,
    'BOGOTA D.C.': 304,
    'CAQUETA': 465,
    'CESAR': 371,
    'CHOCO': 682,
    'CUNDINAMARCA': 419,
    'MAGDALENA': 679,
    'QUINDIO': 362,
    'SUCRE': 269
}

diesel_desv = {
    'ANTIOQUIA': 365,
    'ATLANTICO': 383,
    'BOGOTA D.C.': 436,
    'CAQUETA': 409,
    'CESAR': 352,
    'CHOCO': 612,
    'CUNDINAMARCA': 336,
    'MAGDALENA': 490,
    'QUINDIO': 401,
    'SUCRE': 269
}

### Desviación estándar
La desviación estándar es una medida de cuánto difiere un conjunto de ellos entre ellos. En un conjunto en el que todos los datos son iguales, la desviación estándar será cero. Entre mayor sea la diferencia entre los datos en un conjunto, mayor será la desviación estándar.

Matemáticamente se define como:

$$\displaystyle \sigma = \sqrt{\sum_{i=1}^N\frac{(x_i-\bar{x})^2}{N-1}}$$

En donde $N$ es el número de datos, $\bar{x}$ es el valor promedio de los datos, y $x_i$ es el dato número $i$.

En distribuciones normales ideales se espera que alrededor del 64% de las observaciones estén en el rango $[\bar{x}-\sigma, \bar{x}+\sigma]$.

<img src="fig/teor-devest.png" alt="Picture title" width="600"/>

En la práctica, es muy raro tener distribuciones normales ideales, pero aún así la desviación estándar es una buena medida para determinar la varianza entre los datos y también para determinar cuándo un dato está demasiado lejos del valor promedio.

Veamos cómo es la distribución de precios del Diesel para el Departamento de Magdalena.

![Picture title](fig/desvest.png)

Claramente esta no es una distribución normal, por eso cuando calculamos la cantidad de datos que hay en el rango $[\bar{x}-\sigma,\bar{x}+\sigma]$ obtenemos un $77\%$, que es mayor al valor esperado teóricamente. 

Sin embargo, vamos a utilizar esta métrica y haremos las siguientes definiciones:

- **Estación Cara (+):** Será aquella estación de servicio para la cual el precio de combustible es mayor al precio promedio más la desviación estándar.
- **Estación Barata (-):** Será aquella estación de servicio para la cual el precio de combsutible es menor al precio promedio menos la desviación estándar.
- **Estación Promedio (o):** Será aquella estación de servicio para la cual el precio de combustible tiene una diferencia con el promedio menor a una desviación estándar.

Desde luego, estas definiciones son completamente arbitrarias, en muchos problemas en la práctica es necesario establecer límites como estos, y será la práctica la que determine qué limites funcionan mejor para resolver un problema determinado.

### Condicionales `if`, `else` y `elif`

#### `if`
Python, como muchos otros lenguajes de programación ofrece un método que permite que cierto código se ejecute únicamente cuando se satisface cierta condición, este es el método `if` y su sintaxis en Python es la siguiente:


**if** *condicion*:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion se cumple

La indentación del código después del `if` es MUY importante. Veamos un ejemplo sencillo:


In [6]:
print("Esto se imprime siempre")

if 4 == 3:
    print("Esto se imprime solo si la condición se satisface")

print("Esto se vuelve a imprimir siempre, está afuera del if")

Esto se imprime siempre
Esto se vuelve a imprimir siempre, está afuera del if


Como condición se puede incluir cualquier operación que arroje como resultado un booleano `True` or `False`. Los siguientes son ejemplos de este tipo de operaciones:

In [7]:
print( 4 == 1 )  # comparación de igualdad
print( 4 < 1 )  # comparación menor que
print( 4 <= 1)  # comparación menor igual que
print( 4 > 1 )  # comparación mayor que
print( 4 >= 1 ) # comparación mayor o igual que
print( 4 in [1,3,7,9,4])  # prueba de pertenencia

False
False
False
True
True
True


#### `else`
Al comando `if` se le puede adjuntar un comando `else` (leer como *si no*) para escribir un código que sólo se ejecutará en caso de la condición propuesta para el `if`. El esquema `if-else` queda escrito en el siguiente formato:

**if** *condicion*:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion se cumple <br>
**else**:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion no se cumple <br>

De nuevo, la relación entre los bloques `if` y `else` se establece únicamente asegurando que tengan la misma indentación.

Veamos un ejemplo:

In [8]:
print("Esto se imprime siempre")

if 4 in [3,2,1,4]:
    print("Esto se imprime solo si la condición se satisface")
else:
    print("Esto se imprime solo si la condición no se satisface")

print("Esto se vuelve a imprimir siempre, está afuera del if")

Esto se imprime siempre
Esto se imprime solo si la condición se satisface
Esto se vuelve a imprimir siempre, está afuera del if


In [9]:
print("Esto se imprime siempre")

if 4 in [3,2,1]:
    print("Esto se imprime solo si la condición se satisface")
else:
    print("Esto se imprime solo si la condición no se satisface")

print("Esto se vuelve a imprimir siempre, está afuera del if")

Esto se imprime siempre
Esto se imprime solo si la condición no se satisface
Esto se vuelve a imprimir siempre, está afuera del if


#### `elif`

Python ofrece un comando adicional para imponer condiciones adicionales en caso de que la condición inicial no se cumple, este es el comando `elif`, que se puede interpretar como una composición de los comandos `else` y `if`. El comando debe ir seguido de una nueva condición, y el código que esté dentro del bloque `elif` solo se ejecutará en caso de que la condición del `if` no se cumpla y la condición del `elif` si se satisfaga. Dentro de un mismo esquema condicional, se pueden incluir varios bloques `elif`. La versión más general de un esquema condicional es entonces:

**if** *condicion_1*:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion_1 se cumple <br>
**elif** *condicion_2*:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion_1 no se cumple <br>
&emsp; pero la condicion_2 sí se cumple <br>
**elif** *condicion_3*:<br>
&emsp; codigo a ejecutar<br>
&emsp; si la condicion_1 no se cumple <br>
&emsp; si la condicion_2 no se cumple <br>
&emsp; pero la condicion_3 sí se cumple <br>
**else**:<br>
&emsp; codigo a ejecutar<br>
&emsp; si ninguna de las condiciones <br>
&emsp; anteriores se cumple <br>


Veamos

In [ ]:
print("Esto se imprime siempre")

if 4 in [3,2,1]:
    print("Esto se imprime solo si la condicion 1 se cumple")
elif 3 in [4,2,1]:
    print("Esto se imprime solo si la condicion 1 no se cumple, pero la condición 2 si")
else:
    print("Esto se imprime solo si no se cumplen las condiciones 1 ni 2")

print("Esto se vuelve a imprimir siempre, está afuera del if")

#### Operación lógica AND `&`
Es importante tener en cuenta que una vez el código entra a uno de los bloques en un esquema `if-elif-else` ya no entrará a los demás. Si queremos, por ejemplo que un código se ejecute si queremos que se satisfagan 2 condiciones, entonces debemos unir las dos condiciones usando el operador `&`:

In [ ]:
print("Esto se imprime siempre")

if (4 in [4,2,1]) & (3 in [4,2,1]):
    print("Esto se imprime solo si las dos condiciones se cumplen")
else:
    print("Esto se imprime solo si no se cumple alguna de las dos condiciones")

print("Esto se vuelve a imprimir siempre, está afuera del if")

#### Operación lógica OR `|`
Si queremos que un código se ejecute si se alguna de las condiciones en un grupo dado, entonces debemos unir las condiciones usando el operador `|`:

In [ ]:
print("Esto se imprime siempre")

if (4 in [4,2,1]) | (3 in [4,2,1]):
    print("Esto se imprime si alguna de las dos condiciones se cumple")
else:
    print("Esto se imprime solo si no se cumple ninguna de las dos condiciones")

print("Esto se vuelve a imprimir siempre, está afuera del if")

#### Ejercicio 1

Escriba una función que determine si un número dado es par o es impar, la salida de la función debe ser la palabra `'par'` si el número es par, la palabra `'impar'` si el número es impar, y la frase `'no es entero'` si el número entregado no es entero.

Para realizar el ejercicio se recomienda usar la función módulo `%`. El resultado de ejecutar `A % B` es el residuo de la división de `A` entre `B`. Si un número es par, el residuo de la división por 2 debe ser igual a 0, mientras que si es impar el residuo es 1. Veamos:

In [14]:
def es_tipo(numero) :
    if( numero%2 == 0):
        return "es par";
    elif (numero%2 == 1) :
        return "es impar";
    else:
        return "no es entero";

In [15]:

print (es_tipo(3 % 2))
print (es_tipo(8 % 2))
print (es_tipo(8.354 % 2))

es impar
es par
no es entero


#### Ejercicio 2
Escriba una función que determine si un número es múltiplo de 2 o de 3, o de ninguno de los dos. La salida de la función debe ser `'múltiplo de 2'`, `'múltiplo de 3'`, `'múltiplo de 2 y de 3'`, o `'no es múltiplo de 2 ni de 3'`, según el caso.

In [37]:
def es_multiplo(numero) :
    if( (numero%2 == 0) & (numero%3 == 0) ):
        return "es múltiplo de 2 y 3";
    elif (numero%2 == 0) :
        return "multiplo de 2";
    elif (numero%3 == 0) :
        return "multiplo de 3";
    else:
        return "no es múltiplo de 2 ni de 3";

In [38]:
print(es_multiplo(6))
print(es_multiplo(7))
print(es_multiplo(8))
print(es_multiplo(3))

es múltiplo de 2 y 3
no es múltiplo de 2 ni de 3
multiplo de 2
multiplo de 3


Más información y ejemplos de cómo usar los bloques `if`, `elif` y `else` se puede encontrar en los siguientes recursos:

- https://docs.python.org/3/tutorial/controlflow.html
- https://www.programiz.com/python-programming/if-elif-else
- https://www.tutorialspoint.com/python/python_if_else.htm 

### Clasificando Estaciones de Servicio

El siguiente será el diagrama de flujo para clasificar las estaciones de servicio:
![Picture title](fig/C2_decision_tree.png)

Y el siguiente será el formato en el que se nos entregará la información de las estaciones de servicio:

In [21]:
# Ejemplo 1 de estación de servicio
EDS1 = {
    'nombre': 'BOMBA UNICA LA PISTA',
    'departamento': 'AMAZONAS',
    'gasolina': 14000,
    'diesel': 13000
}

# ejemplo 2 de estación de servicio
EDS2 = {
    'nombre': 'CENTRO DE SERVICIOS SAN FRANCISCO	',
    'departamento': 'MAGDALENA',
    'gasolina': 8322,
    'diesel': 8020
}

Teniendo en cuenta el formato de las estaciones de servicio, y también la forma en que se nos entrega la información de los valores promedio, construyamos una función que ejecute el diagrama de flujo anterior para hacer la clasificación de la estación de servicio:

In [22]:
gasolina_prom.keys()

dict_keys(['ANTIOQUIA', 'ATLANTICO', 'BOGOTA D.C.', 'CAQUETA', 'CESAR', 'CHOCO', 'CUNDINAMARCA', 'MAGDALENA', 'QUINDIO', 'SUCRE'])

In [24]:
# Función clasificar_EDS
# Esta función tiene las siguientes entradas:
#    - EDS_dicc, diccionario con la información de la estación de servicio
#    - g_prom, diccionario con los valores promedio de gasolina en 10 departamentos
#    - g_desv, diccionario con las desviaciones estandar de gasolina en 10 departamentos
#    - d_prom, diccionario con los valores promedio del diesel en 10 departamentos
#    - d_desv, diccionario con las desviaciones estandar del diesel en 10 departamentos


def clasificar_EDS(EDS_dicc, g_prom, g_desv, d_prom, d_desv):
    
    # Verificamos si el departamento está entre los 10 departamentos de los que disponemos información
    depto = EDS_dicc['departamento']
    if depto in g_prom.keys():
        # aquí entramos si la condición se cumple
        # VERIFICANDO LOS PRECIOS DE LA GASOLINA
        if EDS_dicc['gasolina'] < (g_prom[depto] - g_desv[depto]):
            # entramos aquí si la gasolina es barata
            salida1 = '-'
        elif EDS_dicc['gasolina'] > (g_prom[depto] + g_desv[depto]):
            # entramos aquí si la gasolina es cara
            salida1 = '+'
        else:
            # entramos aquí si la gasolina no es cara ni barata
            salida1 = 'o'
        
        # VERIFICANDO LOS PRECIOS DEL DIESEL
        if EDS_dicc['diesel'] < (d_prom[depto] - d_desv[depto]):
            # entramos aquí si el diesel es barato
            salida2 = '-'
        elif EDS_dicc['diesel'] > (d_prom[depto] + d_desv[depto]):
            # entramos aquí si el diesel es caro
            salida2 = '+'
        else:
            # entramos aquí si el diesel no es caro ni barato
            salida2 = 'o'

        # Cuando ya tenemos los dos análisis combinamos las salidas
        salida = salida1+salida2
        return salida
    else:
        # aquí entramos si la condición no se cumple
        return 'NoData'

Ya estamos listos para hacer la prueba con las estaciones `EDS1` y `EDS2`:

In [25]:
clasificar_EDS(EDS1,gasolina_prom,gasolina_desv,diesel_prom, diesel_desv)

'NoData'

Como no tenemos datos disponibles para el Departamento de Amazonas, no es posible realizar el análisis para la EDS1

In [26]:
clasificar_EDS(EDS2,gasolina_prom,gasolina_desv,diesel_prom, diesel_desv)

'+o'

La EDS2 ofrece precios significativamente altos para el diesel pero el valor de la gasolina es más cercano al promedio.

### Clasificando miles de estaciones de servicio
Ya tenemos construida una función capaz de clasificar estaciones de servicio, nuestro siguiente objetivo es aplicar esa función a un total de casi 5000 estaciones. Estamos de acuerdo en que escribir 5000 líneas iguales de código no es una opción y debemos usar las herramientas que el lenguaje de programación nos ofrece para realizar este task.


#### Ciclos `for`
El comando `for` se usa cuando se desea repetir un bloque de código múltiples veces, además permite que el código se ejecute dando diferentes valores a una variable cada vez. La sintaxis de un ciclo `for` es el siguiente:


**for** *variable* in *colección*:<br>
&emsp; código a ejecutar<br>
&emsp; con *variable* tomando cada uno<br>
&emsp; de los valores en la *coleccion*<br>

Arranquemos con un ejemplo sencillo:


In [ ]:
for numero in [13, 25, 26, 2665, -845]:  # el nombre de la variable numero es completamente arbitrario
    # dentro del for, el codigo se ejecuta
    # dando cada vez un valor diferente a numero
    print(numero)
    #print(funcion_ej1(numero))
    print("---") # Así separamos cada vez que se ejecuta un ciclo

La colección sobre la que se realiza la iteración del código al interior del `for` puede ser un conjunto, una lista, una tupla, o incluso un diccionario. Sin embargo, más adelante conoceremos otros iterables posibles como los arreglos de Numpy, o las Series o DataFrames de Pandas.

Veamos otro ejemplo:

In [27]:
suma = 0
for i in range(200):
    suma = suma + i
print(suma)

19900


El comando `range(N)` crea una lista de números que arrancan en `0` y terminan en `N-1`, de manera que con el código anterior estamos haciendo la suma de todos los números enteros desde el `0` hasta el `199`. Todo en unas pocas líneas de código, el poder de la programación empieza cada vez a ser más evidente.

In [28]:
suma = 0
for i in range(10,200):
    suma = suma + i
print(suma)

19855


Cuando el comando `range(N1,N2)` se usa con dos argumentos, entonces el listado de números a iterar arranca en `N1` y termina en `N2-1`. En el ejemplo anterior estamos sumando todos los números entre `10` y `199`.

In [29]:
suma = 0
for i in range(10,200,5):
    suma = suma + i
print(suma)

3895


Cuando el comando `range(N1,N2,paso)` se usa con dos argumentos, entonces el listado de números a iterar arranca en `N1` y termina en `N2-1`, pero los números se barren usando saltos con un valor de `paso`. En el ejemplo anterior estamos sumando todos los números entre `10` y `199` pero usando un paso de `5`, estamos haciendo entonces el cálculo siguiente:
$$10+15+20+\cdots+190+195 = 3895$$

#### Ejercicio 3:
Calcule el valor de la multiplicación de todos los números entre el 1 y el 20.

In [32]:
multiplicacion = 1
for i in range(1,21):
    multiplicacion = multiplicacion * i
print(multiplicacion)

2432902008176640000


#### Ejercicio 3:
Calcule cuántos numeros enteros positivos (mayores que 0) y menores que 1000 existen que son al mismo tiempo múltiplos de 2 y de 3.

In [39]:
numero_positivos = 0
for i in range(0,1000):
    if( es_multiplo(i) == "es múltiplo de 2 y 3"):
        numero_positivos = numero_positivos + 1
print(numero_positivos)

167


Más información y ejemplos de cómo usar los ciclos `for` se puede encontrar aquí:

- https://wiki.python.org/moin/ForLoop
- https://www.w3schools.com/python/python_for_loops.asp 
- https://docs.python.org/3/tutorial/controlflow.html#for-statements 

### Importando los datos de las estaciones de servicio

Ahora procedemos a importar los valores de las estaciones de servicio, que se encuentran en el archivo `precios_combustible_dic_2020.xlsx` que se encuentra en la carpeta `data` de este caso. Para cargar, y procesar, archivos de Excel la herramienta por excelencia es la librería Pandas de Python, a lo largo de este curso estaremos aprovechando y aprendiendo a trabajar con las múltiples funcionalidades de Pandas.

El primer paso para usar Pandas es importar la librería, esto se logra con el comando `import pandas as pd` que se ejecutó al principio de este Notebook, el `as pd` está definiendo un alias para la librería que en este caso será `pd`. Este alias es muy estándar y sus compañeros de trabajo lo mirarán raro si usara un alias diferente para esta librería.

Carguemos entonces los datos del archivo usando la función `read_excel` de Pandas:

In [40]:
pd.read_excel('data/precios_combustible_dic_2020.xlsx')

,NombreDepartamento,nombrecomercial,biodiesel,gasolina_corriente
0,SUCRE,100 CAR SERVICE - PASOANCHO,8290,8225
1,CHOCO,ACPM WILLY,6780,6980
2,BOLIVAR,AGAS GAS NATURAL VEHICULAR,8280,8235
3,CALDAS,AGROCOMBUSTIBLES DEL CAMPO,9250,9150
4,CALDAS,ALMACEN Y ESTACION DE SERVICIO COTRANSPAEZ,8900,8925
...,...,...,...,...
4487,QUINDIO,VILLA MARIANA. R,8100,8220
4488,CORDOBA,YOROSEÑA,9000,8900
4489,RISARALDA,ZEUSS LA ESTANCIA,8029,8153
4490,META,ZONA REFRESCANTE LA ESTACIÓN,7342,7026


In [41]:
datos_combustible = pd.read_excel('data/precios_combustible_dic_2020.xlsx')

Veamos qué resulta cuando imprimimos la nueva variable que hemos creado

In [42]:
datos_combustible

,NombreDepartamento,nombrecomercial,biodiesel,gasolina_corriente
0,SUCRE,100 CAR SERVICE - PASOANCHO,8290,8225
1,CHOCO,ACPM WILLY,6780,6980
2,BOLIVAR,AGAS GAS NATURAL VEHICULAR,8280,8235
3,CALDAS,AGROCOMBUSTIBLES DEL CAMPO,9250,9150
4,CALDAS,ALMACEN Y ESTACION DE SERVICIO COTRANSPAEZ,8900,8925
...,...,...,...,...
4487,QUINDIO,VILLA MARIANA. R,8100,8220
4488,CORDOBA,YOROSEÑA,9000,8900
4489,RISARALDA,ZEUSS LA ESTANCIA,8029,8153
4490,META,ZONA REFRESCANTE LA ESTACIÓN,7342,7026


Este objeto es un DataFrame y tiene los datos organizados en filas y en columnas. Cada columna tiene un nombre y un tipo de variable que Pandas intenta determinar automáticamente. Para acceder a los nombres y tipos de variable de las columnas podemos usar:

In [43]:
datos_combustible.columns

Index(['NombreDepartamento', 'nombrecomercial', 'biodiesel',
       'gasolina_corriente'],
      dtype='str')

In [44]:
datos_combustible.dtypes

NombreDepartamento      str
nombrecomercial         str
biodiesel             int64
gasolina_corriente    int64
dtype: object

Las filas también están caracterizadas por un índice, al que tenemos acceso de la siguiente manera

In [45]:
datos_combustible.index

RangeIndex(start=0, stop=4492, step=1)

En este caso, los ìndices son los automáticamente generados por Pandas y son simplemente un rango entre 0 y 4492. Para conocer el número total de filas en el DataFrame podemos usar:

In [46]:
print (len(datos_combustible))
print (len(datos_combustible.index))

4492
4492


Para llamar a todos los datos de una columna en un DataFrame usamos la siguiente sintaxis: 

`nombre_dataframe[nombre_columna]`

In [47]:
datos_combustible['biodiesel']

0       8290
1       6780
2       8280
3       9250
4       8900
        ... 
4487    8100
4488    9000
4489    8029
4490    7342
4491    8175
Name: biodiesel, Length: 4492, dtype: int64

In [48]:
type(datos_combustible['biodiesel'])

pandas.Series

Note que la salida de esta función contiene tanto los valores de la columna *biodiesel* como los valores de los indices identificando todas las filas. El resultado de obtener una columna completa es una Serie, un DataFrame se puede ver entonces como una colección de Series. 

In [49]:
print( type (datos_combustible))
print( type (datos_combustible['biodiesel']))
print( type (1))
print( type (True))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
<class 'int'>
<class 'bool'>


Para acceder a todos los datos de una fila usamos también el nombre de la fila, pero con un ligero cambio en sintaxis:

In [50]:
datos_combustible.loc[3]  # fila con índice 3

NombreDepartamento                        CALDAS
nombrecomercial       AGROCOMBUSTIBLES DEL CAMPO
biodiesel                                   9250
gasolina_corriente                          9150
Name: 3, dtype: object

De nuevo, esto es una Serie

In [51]:
type (datos_combustible.loc[3])

pandas.Series

Para acceder a una casilla específica de un DataFrame se debe especificar el índice y la columna:

In [ ]:
datos_combustible.loc[5,'biodiesel']  # valor del biodiesel en la entrada número 5

A continuación un pequeño resumen de cómo obtener información de filas, columnas y casillas en un DataFrame usando los índices y los nombres de las columnas:

![Picture title](fig/C2_1.png)


Ya conociendo un poco mejor la estructura de los DataFrames se nos pueden ocurrir algunas estrategias para barrer sobre las filas de la base de datos. Por ejemplo, podríamos iterar sobre la lista de índices de la base de datos y posteriormente llamar con el índice la fila respectiva. Veamos:

In [52]:
datos_combustible.index[:10]

RangeIndex(start=0, stop=10, step=1)

In [53]:
# iteremos sobre los 10 primeros índices
for indice in datos_combustible.index[:10]:
    # imprimamos los valores el biodiesel
    print(datos_combustible.loc[indice,'biodiesel'])
    

8290
6780
8280
9250
8900
8400
7730
8320
8660
8890


#### Ejercicio 5
Escriba un código que imprima los nombres de las 100 primeras estaciones de servicio en la base de datos.

Esta estrategia funciona, entonces vamos ahora a aplicar nuestra función de clasificación a todas las estaciones de servicio de la base de datos. Los resultados los vamos a almacenar en una lista que se llamará `clasificacion`:

In [54]:
# definimos la lista en la que almacenaremos los resultados

clasificacion = []

for indice in datos_combustible.index:
    # armamos ahora el diccionario que identifica la estación de servicio, esta variable se sobreescribe en cada iteración
    EDS = {
        'nombre': datos_combustible.loc[indice,'nombrecomercial'],
        'departamento': datos_combustible.loc[indice,'NombreDepartamento'],
        'gasolina': datos_combustible.loc[indice,'gasolina_corriente'],
        'diesel': datos_combustible.loc[indice,'biodiesel']
    }

    # una vez está listo el diccionario, le pasamos nuestra función de clasificacion
    resultado = clasificar_EDS(EDS,gasolina_prom, gasolina_desv, diesel_prom, diesel_desv)

    # agregamos el resultado a nuestra lista
    clasificacion.append(resultado)



Veamos qué tenemos en la lista `clasificacion`

In [55]:
clasificacion

['oo',
 'oo',
 'NoData',
 'NoData',
 'NoData',
 'oo',
 'oo',
 'NoData',
 'oo',
 'NoData',
 'NoData',
 'NoData',
 'oo',
 'NoData',
 'oo',
 'oo',
 'oo',
 'NoData',
 'oo',
 'o+',
 'oo',
 'oo',
 'NoData',
 '-+',
 'oo',
 'NoData',
 '--',
 'NoData',
 'NoData',
 'oo',
 'oo',
 'NoData',
 'NoData',
 'NoData',
 '++',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 '+o',
 'NoData',
 'NoData',
 'oo',
 'o+',
 'oo',
 'oo',
 'oo',
 '++',
 'oo',
 '-o',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'oo',
 'oo',
 'oo',
 'NoData',
 'oo',
 'oo',
 'oo',
 'oo',
 'oo',
 'NoData',
 'NoData',
 'NoData',
 'oo',
 'oo',
 '++',
 'o-',
 '--',
 'oo',
 'oo',
 'oo',
 'oo',
 '++',
 'oo',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 '++',
 '++',
 '++',
 '++',
 'oo',
 'oo',
 'oo',
 'oo',
 '++',
 'oo',
 'NoData',
 'oo',
 'o-',
 'oo',
 'NoData',
 'oo',
 'oo',
 'NoData',
 'oo',
 '-o',
 '++',
 'NoData',
 'NoData',
 'NoData',
 'NoData',
 'NoDa

In [56]:
len(clasificacion)

4492

Efectivamente tenemos 4492 resultados correspondientes a las 4492 estaciones de servicio de la base de datos. Sin embargo, así como un listado independiente es difícil interpretar nuestros resultados. En este caso es una buena idea entonces agregar los resultados a una nueva columna de nuestro DataFrame, con Pandas esto es un procedimiento sencillo que se puede lograr así:

In [57]:
datos_combustible['clasificacion'] = clasificacion   # el nombre de la columna es completamente arbitrario

Veamos:

In [58]:
datos_combustible

,NombreDepartamento,nombrecomercial,biodiesel,gasolina_corriente,clasificacion
0,SUCRE,100 CAR SERVICE - PASOANCHO,8290,8225,oo
1,CHOCO,ACPM WILLY,6780,6980,oo
2,BOLIVAR,AGAS GAS NATURAL VEHICULAR,8280,8235,NoData
3,CALDAS,AGROCOMBUSTIBLES DEL CAMPO,9250,9150,NoData
4,CALDAS,ALMACEN Y ESTACION DE SERVICIO COTRANSPAEZ,8900,8925,NoData
...,...,...,...,...,...
4487,QUINDIO,VILLA MARIANA. R,8100,8220,oo
4488,CORDOBA,YOROSEÑA,9000,8900,NoData
4489,RISARALDA,ZEUSS LA ESTANCIA,8029,8153,NoData
4490,META,ZONA REFRESCANTE LA ESTACIÓN,7342,7026,NoData


Podemos hacer un conteo de las diferentes clasificaciones obtenidas en la siguiente manera:

In [59]:
datos_combustible['clasificacion'].value_counts()

clasificacion
oo        2065
NoData    1681
o-         178
++         164
o+         158
-o         104
--          74
+o          62
+-           5
-+           1
Name: count, dtype: int64

In [60]:
2065/len(datos_combustible)

0.4597061442564559

Observamos que de las 4492 estaciones en la base de datos, 1681 corresponden a departamentos de los que no tenemos información sobre el promedio y la desviación estándar de los precios. La clasificación más común es 'oo', que corresponde a precios cercanos al promedio tanto para la gasolina como para el diesel. Con 2065 resultados esta clasificación corresponde al 46% del total de estaciones de servicio con datos de comparación. El número de estaciones con precios significativamente elevados tanto de gasolina como de diesel, '++', es de 164, mientras que el número de estaciones con precios significativamente bajos de los dos combustibles, '--' es de 74. La distribución de precios parece ser entonces asimétrica con una mayor probabilidad de tener estaciones muy caras.

Sólo hay 5 estaciones caras para gasolina y baratas para diesel '+-', mientras que solo hay 1 estación en la base de datos identificadas como baratas para gasolina y caras para diesel '-+'. Esto sugiere que, en general, hay una correlación entre los precios de la gasolina y el diesel en las estaciones.

Finalmente, si quisiéramos compartir nuestro reporte, podríamos exportar nuestros resultados a Excel.

In [ ]:
datos_combustible.to_excel('data/EDS_clasificadas.xlsx')

## Conclusiones

Este caso nos ha brindado mucha información nueva que nos va a servir en el futuro, hemos aprendido sobre cómo usar los condicionales `if`, `elif` y `else` para clasificar y tomar decisiones. También hemos aprendido a usar el ciclo `for` para cumplir tareas repetitivas.

Aprendimos también a importar datos desde Excel a Python y a trabajar con las estructuras DataFrames y Series.

## Respuestas a ejercicios propuestos durante el caso

##### Ejercicio 1 - Posible Respuesta

In [ ]:
def funcion_ej1 ( numero ):
    # primero evaluamos si el número es par
    if numero % 2 == 0:
        salida = 'par'
    # luego evaluamos si el número es impar
    elif numero % 2 ==1:
        salida = 'impar'
    # si ninguna de las dos condiciones se cumple, el número no es entero
    else:
        salida = 'no es entero'
    
    # definimos la salida de la función
    return salida

Probemos nuestra propuesta de solución:

In [ ]:
print( funcion_ej1(167) )
print( funcion_ej1(12546) )
print( funcion_ej1(167/58) )

##### Ejercicio 2 - Posible Respuesta

In [ ]:
def funcion_ej2 ( numero ):
    # primero evaluamos si el número es múltiplo de 2 y de 3
    if (numero % 2 == 0) & (numero % 3 == 0):
        salida = 'múltiplo de 2 y de 3'
    # si no cumple las dos condiciones ahora las miramos individualmente
    elif numero % 2 == 0:
        salida = 'múltiplo de 2'
    elif numero % 3 == 0:
        salida = 'multiplo de 3'
    # si no cumple ninguna de las condiciones anteriores, entonces el número no es múltiplo de 2 ni de 3
    else:
        salida = 'no es múltiplo de 2 ni de 3'

    # definimos la salida de la función
    return salida


Probemos nuestra solución

In [ ]:
print(funcion_ej2( 15 ))
print(funcion_ej2( 90 ))
print(funcion_ej2( 92 ))
print(funcion_ej2( 91 ))
print(funcion_ej2( 101.15 ))

##### Ejercicio 3 - Posible Respuesta

In [31]:
prod = 1

for i in range(1,21): # arrancamos en 1, terminamos en 20
    prod = prod * i
print(prod)

2432902008176640000


##### Ejercicio 4 - Posible Respuesta

In [34]:
conteo = 0

for i in range(1,1000): # arrancamos en 1, terminamos en 999
    if (i % 2 == 0) & (i % 3 == 0):
        conteo = conteo + 1
print(conteo)

166


Hay 166 números entre 1 y 999 que son múltiplos tanto de 2 como de 3.

##### Ejercicio 5 - Posible Respuesta

In [ ]:
# iteremos sobre los 100 primeros índices
for indice in datos_combustible.index[:100]:
    # imprimamos los valores el biodiesel
    print(datos_combustible.loc[indice,'nombrecomercial'])

### Origen de los datos
Precios de Combustibles - MinEnergía <br>
https://www.datos.gov.co/Econom-a-y-Finanzas/Precios-de-Combustibles-MinEnerg-a/7pcy-5vx9 